# 12. Batch-Size Calibration（spec §14.4）

## 学習目標
- 同じ**トークン予算**で有効バッチ（step あたりトークン数）を変え、per-step / per-token / per-wallclock の収束を比較する
- 「step が少ない＝速い」という誤解を、token と wall-clock の両軸で解く

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
CAL = ROOT / "experiments/calibrations"

In [2]:
res = load_json(CAL / "batch_size.json")["results"]
fig = go.Figure()
for r in res:
    c = r["curve"]
    fig.add_trace(go.Scatter(x=[p["tokens"] for p in c], y=[p["val_loss"] for p in c],
                             mode="lines+markers", name=f'{r["effective_tokens"]//1024}K tok/step'))
fig.update_layout(title="val loss vs トークン数（同一2Mトークン予算）",
                  xaxis_title="tokens seen", yaxis_title="val loss", template="plotly_white", height=400)
fig.show()
pd.DataFrame([(r["effective_tokens"], r["n_steps"], r["final_val_loss"], r["wallclock_sec"]) for r in res],
             columns=["eff_tokens","n_steps","final_val_loss","wallclock_s"])

,eff_tokens,n_steps,final_val_loss,wallclock_s
0,8192,244,6.4373,5.45
1,16384,122,6.7640,4.89
2,32768,61,7.3377,4.83
3,65536,30,7.5397,4.70


In [3]:
# per-step で見ると錯覚が起きる：横軸を step にすると大バッチが「1stepあたり」良く見える
fig = go.Figure()
for r in res:
    c = r["curve"]
    fig.add_trace(go.Scatter(x=[p["step"] for p in c], y=[p["val_loss"] for p in c],
                             mode="lines+markers", name=f'{r["effective_tokens"]//1024}K tok/step'))
fig.update_layout(title="同じデータを step 軸で見ると？（軸を変えると印象が反転）",
                  xaxis_title="optimizer step", yaxis_title="val loss", template="plotly_white", height=380)
fig.show()

## Observation / Interpretation / Caveat
- **Observation**: 固定トークン予算では、小さい有効バッチ（=多い step 数）ほど低い val loss に到達。wall-clock はほぼ同等。step 軸で見ると大バッチが「1stepあたり良い」ように錯覚するが、token 軸では逆。
- **Interpretation**: 更新回数が多いほど収束が進む（勾配ノイズが正則化的に働く領域）。ただし大バッチは1step が重いので、時間軸で追うと差が縮む。どの軸で語るかで結論が変わる典型例。
- **Caveat**: 小モデル・2Mトークン・CPU/GPU混在の短期比較。大規模では大バッチが並列効率で有利になる領域があり、この結果を大規模へ外挿してはいけない。LRを固定したため、バッチに応じたLRスケーリング（linear/sqrt則）は未考慮。

**次へ**: [18_architecture_ablation](18_architecture_ablation.ipynb)